# 2.3 推理加速策略

在不改变模型参数的前提下，通过优化推理过程中的计算策略，显著提升推理速度和降低延迟。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

---
## 2.3.1 投机解码（Speculative Decoding）

使用小模型（draft model）快速生成多个候选token，再用大模型（target model）并行验证这些token，接受正确的token、拒绝错误的token。在保持与大模型完全相同输出分布的前提下，显著提升推理速度。

### 基本投机解码实现

In [ ]:
class SimpleLM(nn.Module):
    """简化语言模型"""
    def __init__(self, vocab_size=100, dim=64, n_layers=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)
        layers = []
        for _ in range(n_layers):
            layers.append(nn.Linear(dim, dim))
            layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, x):
        h = self.emb(x)
        h = self.net(h)
        logits = self.head(h)
        return logits

vocab_size = 100
draft_model = SimpleLM(vocab_size, dim=32, n_layers=1)
target_model = SimpleLM(vocab_size, dim=64, n_layers=3)

def speculative_decode(draft, target, prompt, max_new_tokens=20, k=5):
    """投机解码
    draft: 小模型，快速生成k个候选token
    target: 大模型，并行验证候选token
    k: 候选token数
    """
    generated = prompt.clone()
    total_draft = 0
    total_accepted = 0
    n_steps = 0

    while generated.shape[0] < prompt.shape[0] + max_new_tokens:
        n_steps += 1
        draft_tokens = []
        draft_probs = []
        current = generated.clone()

        for _ in range(k):
            with torch.no_grad():
                logits = draft(current.unsqueeze(0))
                probs = F.softmax(logits[0, -1], dim=-1)
                token = torch.multinomial(probs, 1).squeeze()
            draft_tokens.append(token)
            draft_probs.append(probs)
            current = torch.cat([current, token.unsqueeze(0)])
            if current.shape[0] >= prompt.shape[0] + max_new_tokens:
                break

        total_draft += len(draft_tokens)

        with torch.no_grad():
            target_logits = target(current.unsqueeze(0))
            target_probs = F.softmax(target_logits[0], dim=-1)

        n_accepted = 0
        for i, (dt, dp) in enumerate(zip(draft_tokens, draft_probs)):
            tp = target_probs[prompt.shape[0] - 1 + i]
            accept_prob = torch.min(tp[dt] / (dp[dt] + 1e-10), torch.tensor(1.0))
            if torch.rand(1).item() < accept_prob.item():
                n_accepted += 1
                generated = torch.cat([generated, dt.unsqueeze(0)])
            else:
                resampled = torch.multinomial(F.relu(tp - dp), 1).squeeze()
                generated = torch.cat([generated, resampled.unsqueeze(0)])
                break
        else:
            bonus_token = torch.multinomial(target_probs[-1], 1).squeeze()
            generated = torch.cat([generated, bonus_token.unsqueeze(0)])
            n_accepted += 1

        total_accepted += n_accepted

        if generated.shape[0] >= prompt.shape[0] + max_new_tokens:
            break

    acceptance_rate = total_accepted / max(total_draft, 1)
    speedup = (total_accepted + n_steps) / n_steps if n_steps > 0 else 1.0
    return generated, acceptance_rate, speedup, n_steps

prompt = torch.randint(0, vocab_size, (8,))
result, acc_rate, speedup, steps = speculative_decode(
    draft_model, target_model, prompt, max_new_tokens=20, k=5
)

print(f"=== 投机解码效果 ===")
print(f"生成tokens: {result.shape[0] - prompt.shape[0]}")
print(f"推理步数: {steps}")
print(f"接受率: {acc_rate:.2%}")
print(f"理论加速比: {speedup:.2f}x")
print(f"\n投机解码核心: 保持与大模型完全相同的输出分布")
print(f"加速取决于: draft模型与target模型的分布匹配度")

### 自投机解码（Self-Speculative Decoding）

不使用额外的draft模型，而是利用同一模型的浅层（early exit）来生成候选token，节省draft模型的内存开销。

In [ ]:
class EarlyExitLM(nn.Module):
    """带早期退出的语言模型"""
    def __init__(self, vocab_size=100, dim=64, n_layers=4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.exit_heads = nn.ModuleList([nn.Linear(dim, vocab_size) for _ in range(n_layers)])
        self.final_head = nn.Linear(dim, vocab_size)

    def forward(self, x, exit_layer=None):
        h = self.emb(x)
        for i, layer in enumerate(self.layers):
            h = F.relu(layer(h))
            if exit_layer is not None and i == exit_layer:
                return self.exit_heads[i](h)
        return self.final_head(h)

model_ee = EarlyExitLM(vocab_size=100, dim=64, n_layers=4)
prompt = torch.randint(0, 100, (8,))

with torch.no_grad():
    logits_full = model_ee(prompt.unsqueeze(0))
    logits_exit0 = model_ee(prompt.unsqueeze(0), exit_layer=0)
    logits_exit1 = model_ee(prompt.unsqueeze(0), exit_layer=1)
    logits_exit2 = model_ee(prompt.unsqueeze(0), exit_layer=2)

def kl_divergence(p, q):
    return (p * (p.log() - q.log())).sum(dim=-1).mean()

probs_full = F.softmax(logits_full[0, -1], dim=-1)
for i, (name, logits) in enumerate([
    ("Exit@Layer0", logits_exit0),
    ("Exit@Layer1", logits_exit1),
    ("Exit@Layer2", logits_exit2),
]):
    probs_exit = F.softmax(logits[0, -1], dim=-1)
    kl = kl_divergence(probs_exit, probs_full)
    top1_match = (probs_exit.argmax() == probs_full.argmax()).item()
    print(f"{name}: KL散度={kl:.4f}, Top-1匹配={top1_match}")

print(f"\n自投机解码: 用浅层(exit@1)作为draft，深层作为target")
print(f"优势: 无需额外模型，节省内存")
print(f"代价: 浅层与深层分布差异可能较大，接受率较低")

---
## 2.3.2 批量推理优化

### 连续批处理（Continuous Batching）

不等待整个batch完成后再处理新请求，而是在每个迭代步动态插入新请求、移除已完成请求，显著提升GPU利用率。

In [ ]:
class ContinuousBatchScheduler:
    """连续批处理调度器"""
    def __init__(self, max_batch_size=8):
        self.max_batch_size = max_batch_size
        self.active_requests = {}
        self.request_queue = []
        self.completed_requests = []
        self.total_tokens_generated = 0
        self.total_steps = 0

    def add_request(self, req_id, max_tokens=20):
        self.request_queue.append({
            'id': req_id,
            'tokens_generated': 0,
            'max_tokens': max_tokens,
            'done': False
        })

    def step(self):
        """执行一步推理"""
        self.total_steps += 1

        while len(self.active_requests) < self.max_batch_size and self.request_queue:
            req = self.request_queue.pop(0)
            self.active_requests[req['id']] = req

        completed_ids = []
        for req_id, req in self.active_requests.items():
            req['tokens_generated'] += 1
            self.total_tokens_generated += 1
            if req['tokens_generated'] >= req['max_tokens']:
                req['done'] = True
                completed_ids.append(req_id)

        for req_id in completed_ids:
            self.completed_requests.append(self.active_requests.pop(req_id))

    def is_done(self):
        return not self.active_requests and not self.request_queue

    def utilization(self):
        return len(self.active_requests) / self.max_batch_size

scheduler = ContinuousBatchScheduler(max_batch_size=4)

for i in range(10):
    scheduler.add_request(f"req_{i}", max_tokens=np.random.randint(5, 25))

utilizations = []
while not scheduler.is_done():
    scheduler.step()
    utilizations.append(scheduler.utilization())

avg_util = np.mean(utilizations)
throughput = scheduler.total_tokens_generated / scheduler.total_steps

print(f"=== 连续批处理效果 ===")
print(f"总请求数: 10")
print(f"最大batch: 4")
print(f"总推理步数: {scheduler.total_steps}")
print(f"总生成tokens: {scheduler.total_tokens_generated}")
print(f"平均GPU利用率: {avg_util:.2%}")
print(f"每步吞吐: {throughput:.2f} tokens/step")
print(f"\n连续批处理 vs 静态批处理:")
static_steps = sum(req['max_tokens'] for req in scheduler.completed_requests) // 4
print(f"  静态批处理步数(估计): ~{static_steps}")
print(f"  连续批处理步数: {scheduler.total_steps}")
print(f"  效率提升: ~{static_steps/scheduler.total_steps:.2f}x")

---
## 2.3.3 早期退出（Early Exit）

并非所有token都需要经过全部Transformer层。简单token在浅层即可获得足够置信的输出，可提前退出计算，节省计算量。

In [ ]:
class EarlyExitTransformer(nn.Module):
    """带早期退出的Transformer"""
    def __init__(self, n_layers=6, dim=64, n_heads=4, n_classes=10, threshold=0.9):
        super().__init__()
        self.n_layers = n_layers
        self.threshold = threshold
        self.layers = nn.ModuleList()
        self.exit_classifiers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'attn': nn.MultiheadAttention(dim, n_heads, batch_first=True),
                'ffn': nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)),
                'ln1': nn.LayerNorm(dim),
                'ln2': nn.LayerNorm(dim),
            }))
            self.exit_classifiers.append(nn.Linear(dim, n_classes))

    def forward(self, x, return_exit_info=False):
        B, N, C = x.shape
        exit_info = {'exits': [], 'confidences': []}

        for i, layer in enumerate(self.layers):
            h = layer['ln1'](x)
            h, _ = layer['attn'](h, h, h)
            x = x + h
            x = x + layer['ffn'](layer['ln2'](x))

            exit_logits = self.exit_classifiers[i](x.mean(dim=1))
            exit_probs = F.softmax(exit_logits, dim=-1)
            confidence = exit_probs.max(dim=-1).values
            exit_info['exits'].append(i)
            exit_info['confidences'].append(confidence.mean().item())

            if i < self.n_layers - 1 and confidence.min().item() > self.threshold:
                if return_exit_info:
                    exit_info['final_exit'] = i
                    return exit_logits, exit_info
                return exit_logits

        if return_exit_info:
            exit_info['final_exit'] = self.n_layers - 1
            return exit_logits, exit_info
        return exit_logits

model_ee = EarlyExitTransformer(n_layers=6, dim=64, n_heads=4, n_classes=10, threshold=0.8)
x = torch.randn(4, 8, 64)

with torch.no_grad():
    out, info = model_ee(x, return_exit_info=True)

print(f"=== 早期退出效果 ===")
print(f"最终退出层: {info['final_exit']}")
print(f"\n各层置信度:")
for i, (layer, conf) in enumerate(zip(info['exits'], info['confidences'])):
    bar = '█' * int(conf * 50)
    marker = ' ← 退出' if layer == info['final_exit'] else ''
    print(f"  Layer {layer}: {conf:.4f} {bar}{marker}")

compute_ratio = (info['final_exit'] + 1) / model_ee.n_layers
print(f"\n计算量: 仅使用{info['final_exit']+1}/{model_ee.n_layers}层 ({compute_ratio:.0%})")
print(f"节省: {(1-compute_ratio)*100:.0f}%")

print(f"\n--- 不同阈值下的退出行为 ---")
for thresh in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]:
    model_ee.threshold = thresh
    with torch.no_grad():
        _, info = model_ee(x, return_exit_info=True)
    print(f"  阈值={thresh:.2f}: 退出@Layer{info['final_exit']}, 计算{(info['final_exit']+1)/6:.0%}")

### 推理加速方法综合对比

In [ ]:
print(f"{'方法':<25} {'加速比':<15} {'额外内存':<15} {'输出一致性':<15}")
print("-" * 70)
methods = [
    ("投机解码", "2-3x", "draft模型", "完全一致"),
    ("自投机解码", "1.5-2x", "无", "完全一致"),
    ("Medusa多头", "2-3x", "额外预测头", "可调"),
    ("连续批处理", "2-4x", "无", "N/A"),
    ("早期退出", "1.5-3x", "退出分类器", "近似"),
]
for name, speedup, mem, consist in methods:
    print(f"{name:<25} {speedup:<15} {mem:<15} {consist:<15}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. 服务端: 投机解码 + 连续批处理 (最大吞吐)")
print(f"2. 端侧(内存充裕): 投机解码 (draft+target两个模型)")
print(f"3. 端侧(内存紧张): 自投机解码 或 早期退出 (单模型)")
print(f"4. 高并发服务: 连续批处理 (必选)")

### Medusa多头预测

Medusa在模型头部添加多个额外的预测头，每个头独立预测未来不同位置的token，一次前向传播生成多个候选，提高投机解码的接受率。

In [ ]:
class MedusaHead(nn.Module):
    """Medusa预测头：预测未来第k个位置的token"""
    def __init__(self, hidden_dim, vocab_size, k):
        super().__init__()
        self.k = k
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, vocab_size),
        )

    def forward(self, hidden_state):
        return self.head(hidden_state)

class MedusaModel(nn.Module):
    """带Medusa头的语言模型"""
    def __init__(self, vocab_size=1000, hidden_dim=128, n_medusa_heads=3, top_k_per_head=5):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.n_medusa_heads = n_medusa_heads
        self.top_k_per_head = top_k_per_head

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=4, batch_first=True),
            num_layers=2,
        )
        self.lm_head = nn.Linear(hidden_dim, vocab_size)
        self.medusa_heads = nn.ModuleList([
            MedusaHead(hidden_dim, vocab_size, k + 1) for k in range(n_medusa_heads)
        ])

    def forward(self, input_ids):
        h = self.embedding(input_ids)
        h = self.transformer(h)
        base_logits = self.lm_head(h)
        medusa_logits = [head(h) for head in self.medusa_heads]
        return base_logits, medusa_logits

    def generate_candidates(self, input_ids):
        """生成树状候选结构"""
        base_logits, medusa_logits = self.forward(input_ids)
        last_pos_logits = base_logits[:, -1, :]
        base_topk = torch.topk(last_pos_logits, self.top_k_per_head, dim=-1)
        candidates = []
        for i in range(self.top_k_per_head):
            base_token = base_topk.indices[0, i].item()
            base_prob = base_topk.values[0, i].item()
            medusa_tokens = []
            for head_idx, m_logits in enumerate(medusa_logits):
                m_top1 = m_logits[0, -1, :].argmax().item()
                medusa_tokens.append(m_top1)
            candidates.append({
                'base_token': base_token,
                'base_prob': base_prob,
                'medusa_tokens': medusa_tokens,
                'path': [base_token] + medusa_tokens,
            })
        return candidates

medusa = MedusaModel(vocab_size=1000, hidden_dim=128, n_medusa_heads=3, top_k_per_head=5)
medusa.eval()
input_ids = torch.randint(0, 1000, (1, 16))

with torch.no_grad():
    base_logits, medusa_logits = medusa(input_ids)
    candidates = medusa.generate_candidates(input_ids)

print(f"=== Medusa多头预测 ===")
print(f"Medusa头数量: {len(medusa.medusa_heads)}")
print(f"每个头Top-K: {medusa.top_k_per_head}")
print(f"候选路径数: {len(candidates)}")
print(f"最大预测长度: 1(base) + {len(medusa.medusa_heads)}(medusa) = {1+len(medusa.medusa_heads)} tokens")
print(f"\n候选路径示例:")
for i, c in enumerate(candidates[:3]):
    print(f"  路径{i}: base={c['base_token']}, medusa={c['medusa_tokens']}, 总长={len(c['path'])} tokens")

base_params = sum(p.numel() for p in medusa.lm_head.parameters())
medusa_params = sum(p.numel() for p in medusa.medusa_heads.parameters())
total_params = sum(p.numel() for p in medusa.parameters())
print(f"\n参数量:")
print(f"  LM Head: {base_params:,}")
print(f"  Medusa Heads: {medusa_params:,} ({len(medusa.medusa_heads)}个头)")
print(f"  Medusa占比: {medusa_params/total_params*100:.1f}%")
print(f"\nMedusa vs 标准投机解码:")
print(f"  优势: 无需额外draft模型, 一次前向传播生成多候选")
print(f"  劣势: Medusa头需要额外训练, 预测精度不如独立draft模型")
print(f"  接受率: 通常30-60% (取决于任务和头数)")

### 产业级实战：真实语言模型投机解码

使用 HuggingFace transformers 加载真实的 GPT-2 模型作为 target model，GPT-2 small 作为 draft model，实现端到端的投机解码。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

tokenizer = AutoTokenizer.from_pretrained('gpt2')
target_model = AutoModelForCausalLM.from_pretrained('gpt2-medium')
draft_model = AutoModelForCausalLM.from_pretrained('gpt2')
target_model.eval()
draft_model.eval()

if torch.cuda.is_available():
    target_model = target_model.cuda()
    draft_model = draft_model.cuda()

def get_device(model):
    return next(model.parameters()).device

def speculative_decode(target_model, draft_model, tokenizer, prompt, max_new_tokens=30, draft_len=4):
    """产业级投机解码实现
    1. draft模型快速生成draft_len个候选token
    2. target模型一次前向传播验证所有候选
    3. 接受匹配的token，拒绝不匹配的并重采样"""
    device = get_device(target_model)
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    generated = input_ids
    total_tokens = 0
    draft_calls = 0
    target_calls = 0
    accepted_tokens = 0

    while total_tokens < max_new_tokens:
        draft_tokens = []
        draft_probs = []
        current = generated

        with torch.no_grad():
            for _ in range(draft_len):
                outputs = draft_model(current)
                next_token_logits = outputs.logits[:, -1, :]
                probs = torch.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                draft_tokens.append(next_token)
                draft_probs.append(probs)
                current = torch.cat([current, next_token], dim=-1)
                draft_calls += 1

        all_tokens = torch.cat(draft_tokens, dim=-1)
        verify_input = torch.cat([generated, all_tokens], dim=-1)

        with torch.no_grad():
            target_outputs = target_model(verify_input)
            target_logits = target_outputs.logits
            target_calls += 1

        n_accepted = 0
        for i in range(draft_len):
            pos = generated.shape[1] - 1 + i
            target_prob = torch.softmax(target_logits[0, pos], dim=-1)
            draft_prob = draft_probs[i][0]
            token = draft_tokens[i][0].item()

            r = torch.rand(1, device=device).item()
            if r < target_prob[token].item() / max(draft_prob[token].item(), 1e-10):
                n_accepted += 1
            else:
                adjusted = torch.clamp(target_prob - draft_prob, min=0)
                if adjusted.sum() > 0:
                    adjusted = adjusted / adjusted.sum()
                    corrected_token = torch.multinomial(adjusted.unsqueeze(0), num_samples=1)
                else:
                    corrected_token = torch.argmax(target_prob).unsqueeze(0).unsqueeze(0)
                draft_tokens[i] = corrected_token
                break

        accepted_tokens_this_round = n_accepted + (1 if n_accepted < draft_len else 0)
        accepted_tokens += accepted_tokens_this_round

        new_tokens = torch.cat([t for t in draft_tokens[:n_accepted + (1 if n_accepted < draft_len else 0)]], dim=-1)
        generated = torch.cat([generated, new_tokens], dim=-1)
        total_tokens += new_tokens.shape[-1]

        if n_accepted == draft_len:
            bonus_token = torch.argmax(target_logits[0, -1], dim=-1).unsqueeze(0).unsqueeze(0)
            generated = torch.cat([generated, bonus_token], dim=-1)
            total_tokens += 1

    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    accept_rate = accepted_tokens / (draft_calls + target_calls) if (draft_calls + target_calls) > 0 else 0
    return text, total_tokens, draft_calls, target_calls, accept_rate

def autoregressive_decode(model, tokenizer, prompt, max_new_tokens=30):
    device = get_device(model)
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompt = 'The future of artificial intelligence is'

start = time.perf_counter()
spec_text, n_tokens, n_draft, n_target, accept_rate = speculative_decode(
    target_model, draft_model, tokenizer, prompt, max_new_tokens=30, draft_len=4
)
spec_time = time.perf_counter() - start

start = time.perf_counter()
auto_text = autoregressive_decode(target_model, tokenizer, prompt, max_new_tokens=30)
auto_time = time.perf_counter() - start

print(f"=== 产业级投机解码: GPT2-medium(target) + GPT2(draft) ===")
print(f"\n自回归输出: {auto_text}")
print(f"投机解码输出: {spec_text}")
print(f"\n{'指标':<20} {'自回归':<15} {'投机解码':<15}")
print("-" * 50)
print(f"{'延迟(s)':<20} {auto_time:<15.3f} {spec_time:<15.3f}")
print(f"{'target前向次数':<20} {'30':<15} {n_target:<15}")
print(f"{'draft前向次数':<20} {'0':<15} {n_draft:<15}")
print(f"{'接受率':<20} {'-':<15} {accept_rate:<15.2%}")

print(f"\n产业界投机解码实践:")
print(f"1. 框架: vLLM (内置speculative decoding)")
print(f"2. 模型对: Llama-70B(target) + Llama-7B(draft)")
print(f"3. 接受率: 通常60-80% (同系列模型)")
print(f"4. 加速比: 1.5-2.5x (取决于接受率和draft_len)")
print(f"5. 自投机: Early Exit / Medusa (无需额外draft模型)")

del target_model, draft_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()